<a href="https://colab.research.google.com/github/MaggieMKWong/my_first_remote/blob/main/preprocessing_tutorials/CheckFormatRds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
if (!require("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

The following notebook takes as input a Seurat or SingleCellExperiment object, saves the object in an h5ad file, and verifies whether the format of the h5ad aligns with the requirements of ArchMap. The steps that are checked include:

(1) adata.X has non-zero count data,\
(2) "batch" is a column in adata.obs,\
(3) and the number of cells does not exceed 200000.

The dataset used in this tutorial can be downloaded [here](https://datasets.cellxgene.cziscience.com/04aca54d-6bb6-4448-ade0-371fa845812d.rds). If you would like to run this notebook yourself, or use your own data, make sure to copy this notebook and change the input and output paths in the following cell.

In [ ]:
input_path = "test_data/pbmc3k.RDS"
output_path = "test.h5ad"

In [ ]:
BiocManager::install(version='3.20')
BiocManager::install("basilisk")
BiocManager::install("zellkonverter")
BiocManager::install("scRNAseq")
BiocManager::install("Seurat")

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.20 (BiocManager 1.30.25), R 4.4.2 (2024-10-31)

Old packages: 'tinytex'

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.20 (BiocManager 1.30.25), R 4.4.2 (2024-10-31)

Warning message:
"package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'basilisk'"
Old packages: 'tinytex'

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.20 (BiocManager 1.30.25), R 4.4.2 (2024-10-31)

Warning message:
"

In [ ]:
library(basilisk)
library(scRNAseq)
library(zellkonverter)
library(Seurat)

# Load dataset (Replace this with your Seurat object)
seurat_obj <- readRDS(input_path)  # Load your Seurat object

# Convert Seurat to SingleCellExperiment (SCE) before saving to H5AD
rdata <- as.SingleCellExperiment(seurat_obj)

#### Uncomment the code below if your query is a SingleCellExperiment object ####
# # Load dataset (Change this to your custom query dataset)
# rdata <- SegerstolpePancreasData()

# Save to H5AD
zellkonverter::writeH5AD(rdata, file = output_path, X_name = "counts")

# Define Basilisk environment
my_env <- BasiliskEnvironment(
  pkgname = "basilisk",
  envname = "my_env_name",
  packages = c("anndata==0.11.3", "h5py==3.13.0")
)

# Read back H5AD file inside Basilisk environment and perform checks
roundtrip <- basiliskRun(
  fun = function(output_path) {
    # Import anndata
    anndata <- reticulate::import("anndata")

    # Read H5AD file
    adata <- anndata$read_h5ad(output_path)

    # Check that adata.X has non-zero count data
    if (is.null(adata$X) || sum(adata$X) == 0) {
      stop("Error: adata.X contains only zeros or is NULL. Count data is necessary for the mapping. Please add raw count data (NOT NORMALIZED) to the X attribute.")
    }

    # Check that "batch" is a column in adata.obs
    if (!"batch" %in% colnames(adata$obs)) {
      stop('Error: "batch" column is missing from adata.obs. Please label your batch covariate as batch in the obs dataframe of the anndata object.')
    }

    # Check that number of cells does not exceed 200000
    if (adata$n_obs > 200000) {
      stop('Error: Number of cells in adata exceeds 200 000. You can find more info on how to correctly subset your query for mapping here: https://archmap-docu.readthedocs.io/en/latest/faqs/index.html#my-query-data-has-more-than-the-limit-of-200-000-cells-what-can-i-do.')
    }

    # Print success message
    print("All checks passed! ✅")

    # Return adata (optional)
    return(adata)
  },
  env = my_env,
  output_path = output_path
)

ERROR: Error in checkForRemoteErrors(lapply(cl, recvResult)): one node produced an error: Error: "batch" column is missing from adata.obs. Please label your batch covariate as batch in the obs dataframe of the anndata object.


If you have obtained the error that your data object does not have a batch label, you can run the following to save the batch covariate, representing variation that you want to integrate out, to the batch column. Please rename "donor_id" to the appropriate batch label you have in your data. If you do not have a batch label, the below code will automatically assign all your cells to a single batch.

In [ ]:
# rdata$batch <- rdata$donor_id mapped_batch
if ("donor_id" %in% colnames(rdata) && length(rdata$donor_id) > 0) {
  rdata$batch <- ifelse(is.na(rdata$donor_id) | rdata$donor_id == "", "default_batch", rdata$donor_id)
} else {
  rdata$batch <- "default_batch"
}

zellkonverter::writeH5AD(rdata, file = output_path, X_name = "counts")


We will now rerun the check

In [ ]:
# Read back H5AD file inside Basilisk environment and perform checks
roundtrip <- basiliskRun(
  fun = function(output_path) {
    # Import anndata
    anndata <- reticulate::import("anndata")

    # Read H5AD file
    adata <- anndata$read_h5ad(output_path)

    # Check that adata.X has non-zero count data
    if (is.null(adata$X) || sum(adata$X) == 0) {
      stop("Error: adata.X contains only zeros or is NULL. Count data is necessary for the mapping. Please add raw count data (NOT NORMALIZED) to the X attribute.")
    }

    # Check that "batch" is a column in adata.obs
    if (!"batch" %in% colnames(adata$obs)) {
      stop('Error: "batch" column is missing from adata.obs. Please label your batch covariate as batch in the obs dataframe of the anndata object.')
    }

    # Check that number of cells does not exceed 200000
    if (adata$n_obs > 200000) {
      stop('Error: Number of cells in adata exceeds 200 000. You can find more info on how to correctly subset your query for mapping here: https://archmap-docu.readthedocs.io/en/latest/faqs/index.html#my-query-data-has-more-than-the-limit-of-200-000-cells-what-can-i-do.')
    }

    # Print success message
    print("All checks passed! ✅")

    # Return adata (optional)
    return(adata)
  },
  env = my_env,
  output_path = output_path
)

All checks have now passed and we can now upload our data to ArchMap 🔥